<!-- The badge's branch, the setup cell's `git clone -b` and the folders it copies the cases from must name the same branch. -->

> **Recommended: run this notebook in Google Colab, nothing to install.**
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/dev/docs/notebooks/notebook_3/notebook_3.ipynb)
>
> The notebook opens with the results of an earlier run. To start from a clean notebook, choose Edit → Clear all outputs in Colab's menu first.
>
> Click the ▶ button on each cell (or press Shift+Enter) to see the results and move on to the next one.
>
> If you can't or don't want to use Colab, the documentation website shows the pre-run results.

# Session 3: How a Navigate model is built

> **Disclaimer**
>
> Navigate is an open-source analytical model intended for research and scenario analysis of maritime decarbonisation pathways. The model, its assumptions and any outputs are provided for illustrative and analytical purposes only.
>
> Model outputs depend on the assumptions selected by the user and should not be interpreted as forecasts, benchmarks, recommendations or commercially optimal outcomes, nor as legal, financial or investment advice.
>
> Users are responsible for selecting and adapting assumptions appropriate to their own circumstances and for exercising their own independent judgement when interpreting or relying on any model outputs.

To have a full understanding of Navigate's structure and language, we recommend working through the [Tutorials](https://zerocarbonshipping.github.io/navigate-zcs/tutorials/index.html) and having a look at the [Reference Manual](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/index.html) before starting this session.

This session opens up the case of [Session 1](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/dev/docs/notebooks/notebook_1/notebook_1.ipynb) (the copper trade) or of [Session 2](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/dev/docs/notebooks/notebook_2/notebook_2.ipynb) (the global case): how its deck is put together from Navigate's library of components, where each number it uses is written, and how Navigate reads those numbers. It uses the tools of Sessions 1 and 2: the deck's own text files, `navigate` to run it and [Lantern](https://github.com/zerocarbonshipping/lantern-zcs/tree/dev) to plot what the run wrote, with polars to read the results file directly. [Session 4](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/04-how-navigate-solves.ipynb) shows how the model solves the case step by step, and its results in detail.

## What you'll do

**Pick the case and run it.** The copper trade of Session 1 or the global case of Session 2, with a log of every file Navigate reads.

**Build the simulation.** See how the case is put together from Navigate's library of components, and follow one fleet down to the numbers it rests on.

**Explore the input data.** Look at the numbers that drive the case, in the files that hold them and in the results file, where Navigate writes some of them back as it read them.

## STEP 1: Setup

Run the two cells below on Colab or [locally](https://zerocarbonshipping.github.io/navigate-zcs/staging/notebooks/index.html#running-a-notebook-on-your-own-machine). The first sets up the Navigate model and Lantern, and copies the cases of Sessions 1 and 2 into this session's folder, `session_3`, next to `plots.yaml`, the plots this session draws. The second defines a helper that shows Lantern's plots in the notebook.

In [1]:
ROOT = "/content"
# ROOT = "/tmp/notebook"  # running locally: any folder of your own, without spaces in its path
%cd {ROOT}

!git clone -q --depth 1 -b dev https://github.com/zerocarbonshipping/navigate-zcs {ROOT}/navigate-zcs
%pip install -q {ROOT}/navigate-zcs \
    git+https://github.com/zerocarbonshipping/lantern-zcs@dev
LIBRARY = f"{ROOT}/navigate-zcs/assumptions"
%env ASSUMPTIONS_DATA_DIR={LIBRARY}

import shutil

notebooks = f"{ROOT}/navigate-zcs/docs/notebooks"
shutil.copytree(f"{notebooks}/notebook_1/copper_trade", f"{ROOT}/session_3/copper_trade", dirs_exist_ok=True)
shutil.copytree(f"{notebooks}/notebook_2/global_case", f"{ROOT}/session_3/global_case", dirs_exist_ok=True)
shutil.copy(f"{notebooks}/notebook_3/plots.yaml", f"{ROOT}/session_3/plots.yaml")
%cd {ROOT}/session_3

/tmp/claude-1000/-home-julius-repos-navigate-zcs/61d9727a-7595-480b-b33c-cb02eb46f88d/scratchpad/nbroot



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
env: ASSUMPTIONS_DATA_DIR=/tmp/claude-1000/-home-julius-repos-navigate-zcs/61d9727a-7595-480b-b33c-cb02eb46f88d/scratchpad/nbroot/navigate-zcs/assumptions


/tmp/claude-1000/-home-julius-repos-navigate-zcs/61d9727a-7595-480b-b33c-cb02eb46f88d/scratchpad/nbroot/session_3


In [2]:
# notebook helper: show the pages Lantern writes in the notebook
import html
from pathlib import Path

from IPython.display import HTML, display

def show(*pages, height=360):
    # the pages side by side, each in a frame of its own, height pixels tall
    frames = ""
    for page in pages:
        with open(page, encoding="utf-8") as file:
            frames += f'<iframe srcdoc="{html.escape(file.read())}" style="flex: 1 1 320px; height: {height}px; border: 0"></iframe>'
    display(HTML(f'<div style="display: flex; flex-wrap: wrap">{frames}</div>'))

---

## STEP 2: Pick the case and run it

Pick the case with `CASE`: `"copper_trade"`, the copper trade of Session 1, or `"global_case"`, the mid regulation scenario of Session 2. The cell then runs it as its session does: the copper trade in a few seconds, the global case on its coarse time grid of 8 time steps in a minute or two.

`-l DEBUG` asks Navigate to write into the run's log every file it reads and every node it takes from the assumptions library: STEP 3 reads the deck through that log. STEP 4 reads the results file, where Navigate writes some of the input numbers back as it read them.

To switch case, change `CASE` and run the cells from here again.

In [3]:
# THE CASE: "copper_trade", the copper trade of Session 1, or "global_case", the global case of Session 2
CASE = "copper_trade"

# each case's deck, the options its session runs it with, and the log and results file the run writes: next to
# the deck, or, for a run with --set, in the logs/ and parquets/ folders beside it
CASES = {
    "copper_trade": {"deck": "copper_trade/example_4.nav", "options": "",
                     "log": "copper_trade/example_4.log",
                     "results": "copper_trade/example_4.parquet"},
    "global_case": {"deck": "global_case/basecase_mid_regulation.nav", "options": "--set GRID=coarse",
                    "log": "global_case/logs/basecase_mid_regulation.log",
                    "results": "global_case/parquets/basecase_mid_regulation.parquet"},
}
DECK = CASES[CASE]["deck"]
OPTIONS = CASES[CASE]["options"]
LOG = CASES[CASE]["log"]
RESULTS = CASES[CASE]["results"]
RUN = Path(RESULTS).stem  # the run's name, which Lantern files its pages under

# solve the case; -l DEBUG logs every file Navigate reads and every node it takes from the library
!navigate {DECK} {OPTIONS} -l DEBUG

——————————————————————————————————————————————————————————————————————————————————————————————————
            |    |    |                                  .oO
           )_)  )_)  )_)                             __/___                  _    _
          )___))___))___)\                     _____/______|              __|_|__|_|__
         )____)____)_____)\\           _______/_____\_______\_____      _|____________|__
       _____|____|____|____\\\__       \              < < <       |    |o o o o o o o o /
~~~~~~~\                   /~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
        ´´´´´´´´´´´´´´´´´´´              Navigate
——————————————————————————————————————————————————————————————————————————————————————————————————
        Navigate - an integrated assessment model for decarbonizing shipping
        Developed by Fonden Mærsk Mc-Kinney Møller Center for Zero Carbon Shipping
        Version 1.1.0.dev0 - for questions, contact navigate@zerocarbonshipping.c

Date: 2026-01-01


Date: 2027-01-01


Date: 2028-01-01


Date: 2029-01-01


Date: 2030-01-01


... (17 outputs left out)

Date: 2048-01-01


Date: 2049-01-01


Date: 2050-01-01


Finished simulation, elapsed time: 0m and 4s.


---

## STEP 3: Build the simulation

This section is a "fast track" version of the [Tutorials](https://zerocarbonshipping.github.io/navigate-zcs/tutorials/index.html). We recommend working through them first.

<details>
<summary>Summary from the tutorials</summary>

Navigate is a bit like a "drag and drop" model: you pick some components (*nodes*) from the ones available and add them to your simulation (*deck*).

A **node** is one specific component of the model: a fleet, a vessel, a route, a port, a fuel plant, a regulation, and so on. Ready-made components are stored in the assumptions library, in `assumptions/defaults/`, one folder per type of node (`Fleet`, `Vessel`, `Port`, ...).

**Modules** are lists of components that can be added all at once: for example, `Load DefaultFleet` (`assumptions/modules/default_fleet.inc`) adds all 23 fleets of the library.

A **deck** is the input to a Navigate simulation: one `.nav` file plus the `.inc` files it includes. The `.nav` file is a table of contents with two sections: `DEFINE` for the nodes the simulation is made of, and `EVENTS` for the timeline it steps through.

Two commands in the `.nav` file add components to the deck. `Load` adds a ready-made module from the assumptions library, by its name (`Load DefaultCalibration`). `Include` adds one of the deck's own `.inc` files, by its path (`Include "includes/fleet.inc"`). A single node can also be taken from the library with `Import`, or copied under a new name with `Copy`.

In the deck, a node is written as a type (e.g. `Fleet`), a name in quotes (e.g. `"bulk_carrier_80k_dwt"`), and a block of attributes in braces. For example, the `Fleet` node `bulk_carrier_80k_dwt` declares the Panamax bulk carriers, the ships both cases have. An attribute can be a number (`InitialVessels`, the number of ships in the first year), a list of numbers (`InitialSplit`, the share of each vessel type in the first year), or a link to other nodes (`Vessels`, `TradeGrowth`).

</details>

Run the next cell to read the case's deck: its `.nav` file, then, from the run's log, the files Navigate read for it, in the order it read them.

In [4]:
# the case's deck: its .nav file
!cat {DECK}

# the files Navigate read for it, from the log: a module of the library for each Load, a file of the case for
# each Include
!grep -E "\] (Load|Include) " {LOG} | sed -e "s|.*_collect: ||" -e "s|{ROOT}/||g"

# SPDX-FileCopyrightText: 2026 Fonden Mærsk Mc-Kinney Møller Center for Zero Carbon Shipping
# SPDX-License-Identifier: CC-BY-4.0

# EXAMPLE 4 - Panamax fleet, Chile to Rotterdam.

DEFINE {
    Load DefaultModelDefinition

    # ---- fleet ----
    Include "includes/fleet.inc"

    # ---- route ----
    Include "includes/route.inc"
    Include "includes/ports.inc"

    # ---- fuel ----
    Load DefaultSource
    Load DefaultTransport
    Include "includes/fuel_production.inc"
    Include "includes/regions.inc"
    Include "includes/producer.inc"
    Include "includes/fuel_delivery.inc"
    Load DefaultEmission

    # ---- regulation ----
    Include "includes/eu_ets.inc"
    Include "includes/fuel_eu.inc"

    # ---- behavioural traits ----
    Load DefaultCalibration
}

EVENTS {
    Load DefaultTimeStepYearly
}


[DEFINE] Load DefaultModelDefinition from navigate-zcs/assumptions/modules/default_model_definition.inc
[DEFINE] Include session_3/copper_trade/includes/fleet.inc
[DEFINE] Include session_3/copper_trade/includes/route.inc
[DEFINE] Include session_3/copper_trade/includes/ports.inc
[DEFINE] Load DefaultSource from navigate-zcs/assumptions/modules/default_source.inc
[DEFINE] Load DefaultTransport from navigate-zcs/assumptions/modules/default_transport.inc
[DEFINE] Include session_3/copper_trade/includes/fuel_production.inc
[DEFINE] Include session_3/copper_trade/includes/regions.inc
[DEFINE] Include session_3/copper_trade/includes/producer.inc
[DEFINE] Include session_3/copper_trade/includes/fuel_delivery.inc
[DEFINE] Load DefaultEmission from navigate-zcs/assumptions/modules/default_emission.inc
[DEFINE] Include session_3/copper_trade/includes/eu_ets.inc
[DEFINE] Include session_3/copper_trade/includes/fuel_eu.inc
[DEFINE] Load DefaultCalibration from navigate-zcs/assumptions/modules/def

A module mostly `Import`s nodes, and a deck can name a node it does not write itself: Navigate reads each such node from the library's `defaults/` folder, which holds one file per node, in a folder named after the node's type, the file named after the node. `Forecast("electricity_cost_europe_mid")` is in `assumptions/defaults/Forecast/electricity_cost_europe_mid.inc`.

Run the next cell to list the library's folders and count the nodes Navigate read from them for this case, by type. It then lists every file the run read as `FILES`, for the cells below to search. On Colab, every file can also be opened from the Files panel on the left (the 📁 folder icon): the cases under `session_3/`, the library under `navigate-zcs/assumptions/`.

In [5]:
# the library: a folder per node type, a file per node
!ls {LIBRARY}/defaults

# the nodes Navigate read from it for this case, by type, from the log
!grep -oE '[A-Za-z]+\("[^"]+"\) was retrieved from the defaults' {LOG} | sort -u | cut -d '(' -f 1 | uniq -c

# every file the run read, from the log, as paths from ROOT: the files it loaded or included, then the library
# file of each node it read from the library
import os
import re

log = Path(LOG).read_text(encoding="utf-8")
files = re.findall(r"(?:from|Include) (\S+\.inc)$", log, re.MULTILINE)
files += [f"{LIBRARY}/defaults/{kind}/{name}.inc"
          for kind, name in re.findall(r'(\w+)\("([^"]+)"\) was retrieved from the defaults', log)]
FILES = " ".join(dict.fromkeys(os.path.relpath(file, ROOT) for file in files))
print(f"The run read {len(FILES.split())} files.")

Converter  Feedstock  Fuel   Process   Regulation  Surface     Timetable  Vessel
Curve	   Fleet      Plant  Producer  Route	   Tank	       Transport
Emission   Forecast   Port   Region    Source	   Technology  Variable


      9 Converter
      9 Curve
      3 Emission
      3 Feedstock
      1 Fleet
     79 Forecast
      5 Fuel
      3 Plant
      6 Process
      1 Region
      2 Source
      1 Surface
      5 Tank
      7 Technology
      8 Timetable
      1 Transport
     60 Variable
      4 Vessel


The run read 222 files.


Nodes refer to each other, forming a nested structure: the `Fleet` lists its `Vessel` nodes, each vessel names its `Tank` nodes (as well as its `Route` and its engines, the `Converter` nodes) and links to the nodes that hold its CAPEX and OPEX.

The nested structure ends with a numerical assumption. A numerical assumption is either written in the attribute itself (`InitialSplit = [0.000, 0.001, 0.000, 0.999]`) or held in a "numerical assumption node" that the attribute links to: a `Variable`, `Curve`, `Forecast`, `Surface` or `Timetable` (see STEP 4).

Run the next cell to follow the nested structure for one of the case's fleets, `FLEET`. It first searches the files the run read for the fleet: the library writes it, and a case's own files can change it after importing it, as the copper trade's `includes/fleet.inc` does to set its number of ships.

In [6]:
# THE FLEET to follow: the copper trade has only bulk_carrier_80k_dwt, the global case the 23 fleets of DefaultFleet
FLEET = "bulk_carrier_80k_dwt"

# where the files the run read write the fleet, import it or change it
!cd {ROOT} && grep -n 'Fleet "{FLEET}"' {FILES}

# then three files of the library, each under its name (tail -n +1 prints them whole):
# - the fleet: Vessels lists its ship types, one for each fuel they can burn
# - its ship that burns LNG: its Capex adds two Forecasts, Tanks names its tanks
# - the first Forecast of that Capex: the hull, in USD, from 2025 to 2050
!cd {LIBRARY}/defaults && tail -n +1 Fleet/{FLEET}.inc Vessel/{FLEET}_ice_methane.inc Forecast/{FLEET}_capex.inc

session_3/copper_trade/includes/fleet.inc:12:Import Fleet "bulk_carrier_80k_dwt"
session_3/copper_trade/includes/fleet.inc:22:Fleet "bulk_carrier_80k_dwt" {
navigate-zcs/assumptions/defaults/Fleet/bulk_carrier_80k_dwt.inc:4:Fleet "bulk_carrier_80k_dwt" {


==> Fleet/bulk_carrier_80k_dwt.inc <==
# SPDX-FileCopyrightText: 2026 Fonden Mærsk Mc-Kinney Møller Center for Zero Carbon Shipping
# SPDX-License-Identifier: CC-BY-4.0

Fleet "bulk_carrier_80k_dwt" {

    # vessel types		
	Vessels = [
		Vessel("bulk_carrier_80k_dwt_ice_ammonia"),
		Vessel("bulk_carrier_80k_dwt_ice_methane"),
 		Vessel("bulk_carrier_80k_dwt_ice_methanol"),
		Vessel("bulk_carrier_80k_dwt_ice_oil")
	]
	
	# initial conditions
	InitialVessels = Variable("initial_vessels_bulk_carrier_80k_dwt")
	
	InitialSplit = [
		0.000,
		0.001,
		0.000,
		0.999
	]
	
	InitialAgeDistribution = [
		0,
		Curve("age_dist_bulk_carrier_80k_dwt_methane"),
		0,
		Curve("age_dist_bulk_carrier_80k_dwt_oil")
	]
	
	
	# trade growth
	TradeGrowth = Forecast("trade_growth_bulk_carrier")
	
	# orderbook
	Orderbooks = [
		Forecast("orderbook_bulk_carrier_80k_dwt_ammonia"),
		Forecast("orderbook_bulk_carrier_80k_dwt_methane"),
		Forecast("orderbook_bulk_carrier_80k_dwt_methanol"),
		Forecast("orderbook_bulk

---

## STEP 4: Explore the input data

### General introduction to Navigate numerical assumptions

Most numerical assumptions in the library are written into a `.inc` file as one
of five node types. A `Variable` is a single number. The other four are lookup
tables, from which the model reads a value when it needs one: a `Curve` is looked
up by one number, a `Forecast` by a date, a `Surface` by two numbers, and a
`Timetable` by a date and a number.

The default numbers in the `assumptions` folder were collected by the Center.
Extended documentation of the underlying assumptions is under development. The
unit of a number depends on the attribute that uses it: the Reference Manual gives
it for each attribute, and many library files also state it in a `# Unit:` line.

<details>
<summary>Using your own numbers instead of the Center's</summary>

To change a number for one simulation, `Import` its node in the deck and reassign
the attributes to change, as Session 2's `levers.inc` does for the fuel prices. To
change the defaults for every simulation, keep your own copy of the `assumptions`
folder and give it to Navigate with `-d` (or `ASSUMPTIONS_DATA_DIR`).

</details>

See the Reference Manual for
[Variable](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/variable.html),
[Curve](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/curve.html),
[Forecast](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/forecast.html),
[Surface](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/surface.html)
and
[Timetable](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/timetable.html)
for more details.

Run the next cell to see one node of each of the five types, all of them read by both cases.

In [7]:
# one node of each type, from the library, each file under its name:
# - a Variable, a single number: the cost of capital a ship burning oil, LNG or methanol is financed at
# - a Curve, looked up by one number: the share of the Panamax bulk carriers burning oil at each age, in years
# - a Forecast, looked up by a date: the cost of electricity in Europe
# - a Surface, looked up by two numbers: the propulsion power of a Panamax bulk carrier, in MW, by its speed in
#   knots (rows) and its cargo utilisation (columns: 0 in ballast, 1 laden)
# - a Timetable, looked up by a date and a number: what a hydrogen plant costs to build, by its size in tons a day
!cd {LIBRARY}/defaults && tail -n +1 Variable/cost_of_capital_vessel.inc Curve/age_dist_bulk_carrier_80k_dwt_oil.inc     Forecast/electricity_cost_europe_mid.inc Surface/bulk_carrier_80k_dwt_propulsion_load.inc     Timetable/hydrogen_electro_alkaline_capex.inc

==> Variable/cost_of_capital_vessel.inc <==
# SPDX-FileCopyrightText: 2026 Fonden Mærsk Mc-Kinney Møller Center for Zero Carbon Shipping
# SPDX-License-Identifier: CC-BY-4.0

# Description: Cost of capital for financing a MF-oil, DF-methane, and DF-methanol vessel.
# Unit: Fraction / year

Variable "cost_of_capital_vessel" {
	Value = 0.0889
}

==> Curve/age_dist_bulk_carrier_80k_dwt_oil.inc <==
# SPDX-FileCopyrightText: 2026 Fonden Mærsk Mc-Kinney Møller Center for Zero Carbon Shipping
# SPDX-License-Identifier: CC-BY-4.0

Curve "age_dist_bulk_carrier_80k_dwt_oil" {
	Table = [
		0,	0.04;
		1,	0.044;
		2,	0.038;
		3,	0.035;
		4,	0.048;
		5,	0.046;
		6,	0.024;
		7,	0.036;
		8,	0.037;
		9,	0.041;
		10,	0.052;
		11,	0.077;
		12,	0.106;
		13,	0.085;
		14,	0.054;
		15,	0.025;
		16,	0.024;
		17,	0.026;
		18,	0.034;
		19,	0.03;
		20,	0.025;
		21,	0.008;
		22,	0.017;
		23,	0.032;
		24,	0.015;
	]
}

==> Forecast/electricity_cost_europe_mid.inc <==
# SPDX-FileCopyrightText: 2026 Fonden Mærsk Mc-K

### The results file

STEP 2's run wrote its results to one Parquet file, a long table with a row for each series and date (see [Results file](https://zerocarbonshipping.github.io/navigate-zcs/reference_manual/results_file.html) in the Reference Manual). Lantern plots it, and any Parquet reader reads it; the cells below use polars.

The results file holds the results of the run, not its inputs, but some results are input numbers as Navigate read them: a regulation's price and limit, a producer's feedstock cap, the share of today's ships already fitted with a technology. Others are worked out from several inputs at once, such as what a plant's fuel costs to make. The sections below read both. The file holds the dates of the run's time steps only: on the global case's coarse grid, 8 dates, which its plots join with straight lines.

Run the next cell to read the results file.

In [8]:
import polars as pl

# tables with names in full, and numbers with two decimals and a thousands separator
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(2)
pl.Config.set_thousands_separator(",")

# the results file of STEP 2: a row for each series and date
results = pl.read_parquet(RESULTS)
first, last = results["date"].min(), results["date"].max()
results

kind,node,attribute,key1,key2,date,value
str,str,str,str,str,date,f32
"""global""","""""","""baseline_energy""",null,null,2026-01-01,"33,457,384.00"
"""global""","""""","""baseline_energy""",null,null,2027-01-01,"34,190,100.00"
"""global""","""""","""baseline_energy""",null,null,2028-01-01,"34,938,864.00"
"""global""","""""","""baseline_energy""",null,null,2029-01-01,"35,704,024.00"
"""global""","""""","""baseline_energy""",null,null,2030-01-01,"36,485,944.00"
…,…,…,…,…,…,…
"""vessel""","""bulk_carrier_80k_dwt_ice_ammonia""","""total_fuel_related_expenses""",null,null,2046-01-01,"6,945,452.50"
"""vessel""","""bulk_carrier_80k_dwt_ice_ammonia""","""total_fuel_related_expenses""",null,null,2047-01-01,"6,924,990.00"
"""vessel""","""bulk_carrier_80k_dwt_ice_ammonia""","""total_fuel_related_expenses""",null,null,2048-01-01,"6,911,783.00"


### Exploring the input data specific to the case

The rest of this part checks the numbers that drive this case, in four groups:

1. **Fleet**: what the ships cost to build and to retrofit, and the efficiency technologies
   on offer
2. **Fuels**: the price of the fossil fuels and of electricity, what each alternative
   fuel costs to make, how much feedstock is available, and what it costs to ship a
   fuel from the plant to the bunker port
3. **Regulations**: the prices and limits each regulation sets, its penalty, and what each
   regulation counts
4. **Behavioural traits**: how sharply the owners react to a difference in cost or value, and the producers
   to a difference in demand or cost, and how much of last year's choices the owners and the operators repeat

Each group searches the files the run read, `FILES`, for the attributes and commands that set its numbers, then reads what it can from the results file. To find any other number, do the same: search `FILES` for the attribute, then open the node it names, until you reach the number.

---

### 1. Data about the fleet

The most important data that will affect how the fleet ends up being designed are:
- the ship retrofitting costs from one fuel to another
- the CAPEX of newbuild ships and their financing conditions
- the efficiency technology on offer, and what it costs

The fleet design also depends on the composition of the initial fleet (`InitialVessels`, `InitialSplit` and `InitialAgeDistribution`), the efficiency technology already installed, how fast the trade grows (`TradeGrowth`), when ships are scrapped (`Lifetime`, `FixedScrapRate`), and the ship orderbook (`Orderbooks`, the ships already ordered but not yet delivered).

#### Newbuild CAPEX, retrofitting costs and financing conditions

A newbuild is charged in four parts: the vessel's own `Capex`, here the hull plus the fuel supply system (fuel handling and safety), the engines (each `Converter` priced per MW installed) and the tanks (each `Tank` priced per m3 of containment).

Notes:

- The cargo space lost to larger fuel tanks is deliberately not included in the base assumptions: with an optimised ship design it can be reduced to a negligible amount.
- The ammonia ship is financed at a higher cost of capital in the early years, because of the higher risk.

Run the next cell to see the lines of the fleet's vessel files that set its CAPEX, cost of capital and lifetime, then the conversion costs of the fleet's file. Last, from the results file, what a new ship costs its operator per year, the asset charter rate: the CAPEX and fixed OPEX of its four parts, levelized over its operating years at its cost of capital.

In [9]:
# what each of the fleet's ships costs to build and finance, and how long it lasts, in its vessel file
!cd {LIBRARY}/defaults && grep -E "Capex|CostOfCapital|Lifetime" Vessel/{FLEET}_*.inc

# what converting an existing ship to another fuel costs, in USD, in the fleet's file
!grep set_conversion_cost {LIBRARY}/defaults/Fleet/{FLEET}.inc

# what a new ship costs its operator per year, in USD, ordered in the first year and in the last, from the results file
charter = results.filter(pl.col("attribute") == "asset_charter_rate", pl.col("node").str.starts_with(FLEET + "_"))
charter.filter(pl.col("date").is_in([first, last])).pivot(on="date", index="node", values="value")

Vessel/bulk_carrier_80k_dwt_ice_ammonia.inc:	Lifetime = Variable("bulk_carrier_80k_dwt_lifetime")
Vessel/bulk_carrier_80k_dwt_ice_ammonia.inc:	Capex = <Forecast("bulk_carrier_80k_dwt_capex") + Forecast("bulk_carrier_80k_dwt_supply_system_ammonia_capex")>
Vessel/bulk_carrier_80k_dwt_ice_ammonia.inc:	CostOfCapital = Forecast("cost_of_capital_vessel_ammonia")
Vessel/bulk_carrier_80k_dwt_ice_methane.inc:	Lifetime = Variable("bulk_carrier_80k_dwt_lifetime")
Vessel/bulk_carrier_80k_dwt_ice_methane.inc:	Capex = <Forecast("bulk_carrier_80k_dwt_capex") + Forecast("bulk_carrier_80k_dwt_supply_system_methane_capex")>
Vessel/bulk_carrier_80k_dwt_ice_methane.inc:	CostOfCapital = Variable("cost_of_capital_vessel")
Vessel/bulk_carrier_80k_dwt_ice_methanol.inc:	Lifetime = Variable("bulk_carrier_80k_dwt_lifetime")
Vessel/bulk_carrier_80k_dwt_ice_methanol.inc:	Capex = <Forecast("bulk_carrier_80k_dwt_capex") + Forecast("bulk_carrier_80k_dwt_supply_system_methanol_capex")>
Vessel/bulk_carrier_80k_dwt_ice_

	set_conversion_cost("*ice_oil*", "*ice_methanol*", 17.5e6)
	set_conversion_cost("*ice_oil*", "*ice_ammonia*", 24.1e6)
	set_conversion_cost("*ice_methane*", "*ice_ammonia*", 14.2e6)


node,2026-01-01,2050-01-01
str,f32,f32
"""bulk_carrier_80k_dwt_ice_oil""","6,874,061.00","6,874,061.00"
"""bulk_carrier_80k_dwt_ice_methane""","7,754,126.00","7,754,126.00"
"""bulk_carrier_80k_dwt_ice_methanol""","7,036,686.00","7,036,686.00"
"""bulk_carrier_80k_dwt_ice_ammonia""","8,692,976.00","7,812,293.50"


#### Efficiency technology on offer, and what is already fitted

The assumptions library carries efficiency measures for most vessel classes,
seven for the Panamax bulk carrier, each with its own cost and saving. The fleet starts with a share of its
ships already carrying each technology, which `set_initial_technology_share` sets by the ship's age.

Run the next cell to see, from the results file, the share of each of the fleet's ship types carrying each technology in the first year, before the model decides anything. Then, from each technology's file: the share of an energy demand it saves, its life in years (none: it stays until the ship leaves the fleet), its CAPEX and its yearly OPEX, in USD.

In [10]:
# the share of each of the fleet's ship types (columns) carrying each technology (rows) in the first year
uptake = results.filter(pl.col("attribute") == "technology_uptake", pl.col("node") == FLEET, pl.col("date") == first)
display(uptake.pivot(on="key1", index="key2", values="value").rename({"key2": "technology"}))

# the technologies the fleet's file lists, and what each saves and costs, in its own file
fleet = Path(f"{LIBRARY}/defaults/Fleet/{FLEET}.inc").read_text(encoding="utf-8")
technologies = " ".join(f"Technology/{name}.inc" for name in re.findall(r'Technology\("([^"]+)"\)', fleet))
!cd {LIBRARY}/defaults && grep -E "set_energy_saving|set_external_power|Lifetime|Capex|Opex" {technologies}

technology,bulk_carrier_80k_dwt_ice_ammonia,bulk_carrier_80k_dwt_ice_methane,bulk_carrier_80k_dwt_ice_methanol,bulk_carrier_80k_dwt_ice_oil
str,f32,f32,f32,f32
"""air_lubrication_bulk_carrier_panamax""",0.00,0.00,0.00,0.00
"""exhaust_gas_boiler_ae_bulk_carrier_panamax""",0.00,0.50,0.00,0.50
"""flettner_rotor_bulk_carrier_panamax""",0.00,0.00,0.00,0.00
"""hull_painting_bulk_carrier_panamax""",0.00,0.10,0.00,0.10
"""lighting_system_bulk_carrier_panamax""",0.00,0.50,0.00,0.50
"""propeller_improvement_devices_bulk_carrier_panamax""",0.00,0.25,0.00,0.25
"""variable_frequency_drive_bulk_carrier_panamax""",0.00,0.25,0.00,0.25


Technology/air_lubrication_bulk_carrier_panamax.inc:	set_energy_saving(PROPULSION, 0.04)
Technology/air_lubrication_bulk_carrier_panamax.inc:	Lifetime = 25
Technology/air_lubrication_bulk_carrier_panamax.inc:	Capex = 1.5e6
Technology/air_lubrication_bulk_carrier_panamax.inc:	Opex = 20e3
Technology/exhaust_gas_boiler_ae_bulk_carrier_panamax.inc:	set_energy_saving(HEAT, 0.05)
Technology/exhaust_gas_boiler_ae_bulk_carrier_panamax.inc:	Capex = 100e3
Technology/exhaust_gas_boiler_ae_bulk_carrier_panamax.inc:	Opex = 10e3
Technology/flettner_rotor_bulk_carrier_panamax.inc:	set_external_power(PROPULSION, 0.4)
Technology/flettner_rotor_bulk_carrier_panamax.inc:	Capex = 2e6
Technology/flettner_rotor_bulk_carrier_panamax.inc:	Opex = 40e3
Technology/hull_painting_bulk_carrier_panamax.inc:	set_energy_saving(PROPULSION, 0.05)
Technology/hull_painting_bulk_carrier_panamax.inc:	Lifetime = 5.
Technology/hull_painting_bulk_carrier_panamax.inc:	Capex = 200e3
Technology/hull_painting_bulk_carrier_panamax.

---

### 2. Data about the fuels

The most important data related to fuels are:
- the market price of the fuels bought on a liquid market, such as fuel oil and LNG
- the cost of making each alternative fuel: the plant, the electricity and the feedstock it uses
- how much feedstock there is, which limits how much of a fuel can be made
- what it costs to deliver a fuel to the bunker port, and the port's handling cost

The supply also depends on each plant's cost of capital, lead time, lifetime and uptime, and on how much a port can deliver (`set_bunkering_limit`).

In both cases, we assume that fossil fuels (fuel oil and LNG) have a flat price until 2050. Electricity is the largest part of what an e-fuel costs to make. It is assumed that its cost decreases in the future.

Run the next cell to see where the case sets the market price of fuel oil and LNG, the ports' handling cost and the cost of renewable electricity in each region, then the bunker price the ports charge in the first year, from the results file: the market price plus the port's handling cost.

In [11]:
# where the case sets the market price of a fuel, a port's handling cost and the cost of renewable electricity
!cd {ROOT} && grep -n -E 'set_fuel_price|set_handling_cost|set_source_opex\("renewable"' {FILES}

# the market price of fuel oil, in USD/t
!cat {LIBRARY}/defaults/Forecast/fossil_fuel_oil_price_global.inc

# the bunker price of fuel oil and LNG at each port in the first year, in USD/t, from the results file
results.filter(pl.col("attribute") == "bunker_price", pl.col("date") == first,
               pl.col("key1").is_in(["fossil_fuel_oil", "liquefied_natural_gas"]))

session_3/copper_trade/includes/ports.inc:20:    set_fuel_price("fossil_fuel_oil", Forecast("fossil_fuel_oil_price_global"))
session_3/copper_trade/includes/ports.inc:22:    set_fuel_price("liquefied_natural_gas", Forecast("liquefied_natural_gas_price_global"))
session_3/copper_trade/includes/ports.inc:27:    set_handling_cost("fossil_fuel_oil", Variable("handling_cost_oil"))
session_3/copper_trade/includes/ports.inc:28:    set_handling_cost("liquefied_natural_gas", Variable("handling_cost_lng"))
session_3/copper_trade/includes/ports.inc:29:    set_handling_cost("methanol_electro", Variable("handling_cost_methanol"))
session_3/copper_trade/includes/ports.inc:30:    set_handling_cost("methanol_bio", Variable("handling_cost_methanol"))
session_3/copper_trade/includes/ports.inc:31:    set_handling_cost("ammonia_electro", Variable("handling_cost_ammonia"))
session_3/copper_trade/includes/regions.inc:19:    set_source_opex("renewable", Forecast("electricity_cost_europe_mid"))
navigate-zcs/a

# SPDX-FileCopyrightText: 2026 Fonden Mærsk Mc-Kinney Møller Center for Zero Carbon Shipping
# SPDX-License-Identifier: CC-BY-4.0

# Description: Forecast of the global fossil fuel oil price.
# Unit: USD / ton

Forecast "fossil_fuel_oil_price_global" {
	Table = [
		"01-01-2026",	485;
		"01-01-2050",	485;
	]

	Extrapolate = FLAT
}


kind,node,attribute,key1,key2,date,value
str,str,str,str,str,date,f32
"""port""","""san_antonio""","""bunker_price""","""fossil_fuel_oil""",null,2026-01-01,493.00
"""port""","""san_antonio""","""bunker_price""","""liquefied_natural_gas""",null,2026-01-01,451.00
"""port""","""rotterdam""","""bunker_price""","""fossil_fuel_oil""",null,2026-01-01,493.00
"""port""","""rotterdam""","""bunker_price""","""liquefied_natural_gas""",null,2026-01-01,451.00


A fuel is made in a chain of processes (`Process` nodes), for example hydrogen production by electrolysis, then methanol synthesis. Each region sets what each process costs to build and run (`set_process_capex`, `set_process_opex`) and the energy it uses (`set_process_energy`).

The synthesis and electrolyser cost tables are currently assumed to be flat over time. Learning rates (costs falling as more capacity is built) may be added in a future version of Navigate.

A process's CAPEX and fixed OPEX are given in USD per ton/year of nominal production capacity, and depend on the process's size, in tons/day of output: the `Timetable` of STEP 4's introduction is the hydrogen plant's. The plant's full load hours (the hours it runs at full capacity) are set with `Uptime`, as a share of the year: 0.9 for a biofuel plant in the library, such as bio-methanol from gasification, about 7,900 hours.

Run the next cell to see where the case sets what each process costs to build, then, from the results file, what each plant's fuel costs to make: the cost of a plant built in the first year and in the last, its processes, electricity, feedstock and capital levelized over the fuel it makes, in USD/t, cheapest last year first.

In [12]:
# where the case sets what each process costs to build: a Forecast, or a Timetable by the process's size
!cd {ROOT} && grep -n set_process_capex {FILES}

# what each plant's fuel costs to make, in USD/t, for a plant built in the first year and in the last, from the results file
costs = results.filter(pl.col("attribute") == "investment_cost", pl.col("date").is_in([first, last]))
costs.pivot(on="date", index="node", values="value").sort(str(last))

session_3/copper_trade/includes/regions.inc:36:    set_process_capex("electrolysis_stack_alkaline", Timetable("electrolysis_stack_alkaline_capex"))
session_3/copper_trade/includes/regions.inc:42:    set_process_capex("hydrogen_electro_alkaline", Timetable("hydrogen_electro_alkaline_capex"))
session_3/copper_trade/includes/regions.inc:49:    set_process_capex("methanol_synthesis_electro_*", Timetable("methanol_synthesis_electro_capex"))
session_3/copper_trade/includes/regions.inc:56:    set_process_capex("ammonia_synthesis_electro_*", Timetable("ammonia_synthesis_capex"))
session_3/copper_trade/includes/regions.inc:61:    set_process_capex("methanol_synthesis_bio_gasification", Forecast("methanol_synthesis_bio_gasification_capex"))


node,2026-01-01,2050-01-01
str,f32,f32
"""plant_ammonia_electro_alkaline_americas""","1,258.06",718.58
"""plant_methanol_bio_gasification_europe""",962.14,810.15
"""plant_ammonia_electro_alkaline_europe""","1,369.08",833.05
"""plant_methanol_electro_alkaline_americas""","1,596.48","1,012.84"
"""plant_methanol_electro_alkaline_europe""","1,724.64","1,146.40"


#### Feedstock availability

The availability of a fuel made from biomass or biogenic CO2 is limited by its feedstock supply.
For example, bio-methanol needs gasifiable biomass, and e-methanol needs biogenic CO2. Each cap is set on the producer that makes the fuel, with `set_feed_limit`.

Run the next cell to see where the case sets each cap, then draw the caps from the results file with Lantern. Lantern reads the plots it draws from `plots.yaml` in the folder it is started in: this session's lists the input numbers the results file holds, under `inputs`.

In [13]:
# where the case caps the feedstock each producer may use
!cd {ROOT} && grep -n set_feed_limit {FILES}

# the caps as Navigate read them, drawn from the results file with this session's plots.yaml, and shown
!lantern {RESULTS} --scale 0.8 --plot inputs_feed_constraint
show(f"plots/{RUN}/inputs_feed_constraint.html", height=480)

session_3/copper_trade/includes/producer.inc:16:	set_feed_limit("carbon_dioxide_biogenic", <Forecast("carbon_dioxide_biogenic_availability_flat") * Forecast("carbon_dioxide_biogenic_share_shipping") * Variable("trade_share_biogenic_carbon_dioxide")>)
session_3/copper_trade/includes/producer.inc:25:	set_feed_limit("feedstock_biomethanol_gasification", <Forecast("feedstock_biomethanol_gasification_availability") * Forecast("feedstock_biomethanol_gasification_share_shipping") * Variable("trade_share_gasifiable_biomass")>)


Plots from /tmp/claude-1000/-home-julius-repos-navigate-zcs/61d9727a-7595-480b-b33c-cb02eb46f88d/scratchpad/nbroot/session_3/plots.yaml


Loaded example_4: 1 run(s), 3 of 3 plots with data


Plotting:   0%|                                        | 0/1 [00:00<?, ?plot/s]

Plotting: 100%|████████████████████████████████| 1/1 [00:00<00:00, 10.41plot/s]
Wrote 1 plots to plots
plots/example_4/inputs_feed_constraint.html


#### Delivering the fuel to the bunker ports

Shipping a fuel from its plant to the bunker port adds to its cost (`set_fuel_distance` on the `Plant`, at the
rate its region sets with `set_transport_cost`), so fuel is cheap at the port next door and carries a penalty
at the far one. Run the next cell to see where the case sets them.

In [14]:
# where the case sets how far each plant ships its fuel, in nautical miles, and at what cost, in USD per ton and mile
!cd {ROOT} && grep -n -E "set_fuel_distance|set_transport_cost" {FILES} || echo "No plant of this case ships its fuel to the ports at a cost."

session_3/copper_trade/includes/fuel_delivery.inc:15:	set_transport_cost("fuel_shipping", 0.02)
session_3/copper_trade/includes/fuel_delivery.inc:24:	set_fuel_distance("san_antonio", 1250)
session_3/copper_trade/includes/fuel_delivery.inc:25:	set_fuel_distance("rotterdam", 7600)
session_3/copper_trade/includes/fuel_delivery.inc:29:	set_fuel_distance("rotterdam", 540)
session_3/copper_trade/includes/fuel_delivery.inc:30:	set_fuel_distance("san_antonio", 7600)


---

### 3. Data about the regulations

The most important data about a regulation are:
- the **price** it puts on emissions, for a scheme that prices them
- the **limit** it sets, for a scheme that caps the intensity of the fuel
- the **penalty** for missing that limit
- its **scope**: which emissions it counts, with its own emission factors
- **how much of the voyage** it counts: which ports are inside the scheme

Besides these, a regulation sets which ships it covers (`set_include_vessel`) and whether each ship complies alone or the ships pool their compliance (`Scheme`).

Both cases model FuelEU Maritime and the EU ETS as `Regulation` nodes, a yearly target with a penalty for missing it: the global case from the library, the copper trade in its own files. The global case adds a global fuel standard (`gfs`), an intensity limit with a penalty, as FuelEU, on every voyage. Navigate's other policy node, the `Levy`, charges or subsidises a fuel per ton of emissions; neither case uses one. The two EU instruments work differently:

| | FuelEU Maritime | EU ETS |
|:---|:---|:---|
| **What it measures** | the **intensity** of the energy used, gCO2eq/MJ | the **quantity** emitted, tons CO2eq |
| **What you pay** | a penalty on how far you miss the annual target | a price on every covered ton: a target of zero tons, with the allowance price as the penalty |
| **Emissions counted** | **well-to-wake**: how the fuel was made counts | **tank-to-wake**: only what the ship burns on board |

The ports inside a scheme are its `Jurisdiction`. Each leg (a ship's one-way sailing from one port to the next) is counted by a share: `IntraFraction` when both ends are inside, `InterFraction` when one is, `ExtraFraction` when neither; a stay in a port inside counts as `IntraFraction`.

Run the next cell to see how the case sets each regulation's measure, scheme, scope and geography, then its price or penalty (`RemedialCost`) and its limit (`set_vessel_threshold`). It then draws both from the results file with Lantern.

In [15]:
# where the case writes each regulation, and the lines that set what it measures, where and at what price
!cd {ROOT} && grep -n -E '^Regulation |^\s*(Measure|Scheme|Scope|Jurisdiction|IntraFraction|InterFraction|ExtraFraction|RemedialCost) |set_vessel_threshold' {FILES}

# the price of a ton and the intensity limits as Navigate read them, drawn from the results file, and shown
!lantern {RESULTS} --scale 0.8 --plot inputs_remedial_cost --plot inputs_intensity_limit
show(f"plots/{RUN}/inputs_remedial_cost.html", f"plots/{RUN}/inputs_intensity_limit.html")

session_3/copper_trade/includes/eu_ets.inc:16:Regulation "eu_ets" {
session_3/copper_trade/includes/eu_ets.inc:17:	Measure = ABSOLUTE
session_3/copper_trade/includes/eu_ets.inc:18:	Scheme = INDIVIDUAL
session_3/copper_trade/includes/eu_ets.inc:19:	Scope = TTW
session_3/copper_trade/includes/eu_ets.inc:25:	Jurisdiction = [Port("rotterdam")]
session_3/copper_trade/includes/eu_ets.inc:26:	IntraFraction = 1     # both ends in the EU
session_3/copper_trade/includes/eu_ets.inc:27:	InterFraction = 0.5   # one end in the EU
session_3/copper_trade/includes/eu_ets.inc:28:	ExtraFraction = 0     # neither end in the EU
session_3/copper_trade/includes/eu_ets.inc:30:	RemedialCost = <Forecast("eu_ets_carbon_credit_price") * Forecast("eu_ets_maritime_phase_in")>
session_3/copper_trade/includes/eu_ets.inc:33:	set_vessel_threshold("*", 0.)
session_3/copper_trade/includes/fuel_eu.inc:16:Regulation "fuel_eu" {
session_3/copper_trade/includes/fuel_eu.inc:17:	Measure = INTENSITY
session_3/copper_trade/inclu

Plots from /tmp/claude-1000/-home-julius-repos-navigate-zcs/61d9727a-7595-480b-b33c-cb02eb46f88d/scratchpad/nbroot/session_3/plots.yaml


Loaded example_4: 1 run(s), 3 of 3 plots with data


Plotting:   0%|                                        | 0/2 [00:00<?, ?plot/s]

Plotting: 100%|████████████████████████████████| 2/2 [00:00<00:00, 16.89plot/s]
Wrote 2 plots to plots
plots/example_4/inputs_remedial_cost.html
plots/example_4/inputs_intensity_limit.html


#### Regulation emission factors

Navigate's own estimate counts each fuel's emissions well-to-wake: the fuel's emission factors, each engine's GHG slip, and what making and transporting the fuel emitted. A regulation can count the same fuel by its own rules. Besides its scope (well-to-wake or tank-to-wake), a regulation's count differs from Navigate's estimate in three ways:

1. **Its own emission factors**: `set_fuel_ttw` and `set_fuel_wtt` in the regulation's file replace Navigate's
   own where the regulation states them. That is how the EU ETS counts no combustion CO2 for bio- and e-fuels.
2. **Its own warming potentials**: the EU ETS weights methane 28 and nitrous oxide 265, FuelEU 25 and 298;
   Navigate's results use the library's, 29.8 for methane over 100 years.
3. **Its geography**: only the legs and port stays its `Jurisdiction` and fractions cover are counted.

Run the next cell to see the factors each regulation states for e-methanol and its warming potentials (`set_gwp`). Change `FUEL` to see another fuel's.

In [16]:
# THE FUEL whose emission factors to see: fossil_fuel_oil, liquefied_natural_gas, methanol_bio, methanol_electro, ...
FUEL = "methanol_electro"

# the emission factors each regulation states for it, and the warming potentials it weights them by
!cd {ROOT} && grep -n -E 'set_fuel_(ttw|wtt)\("{FUEL}"|set_gwp' {FILES}

session_3/copper_trade/includes/eu_ets.inc:36:	set_gwp("carbon_dioxide", 1)
session_3/copper_trade/includes/eu_ets.inc:37:	set_gwp("methane", 28)
session_3/copper_trade/includes/eu_ets.inc:38:	set_gwp("nitrous_oxide", 265)
session_3/copper_trade/includes/eu_ets.inc:51:	set_fuel_ttw("methanol_electro", "carbon_dioxide", 0)
session_3/copper_trade/includes/eu_ets.inc:52:	set_fuel_ttw("methanol_electro", "methane", <Variable("eu_default_diesel_ice_methane") * Forecast("eu_ets_ghg_phase_in")>)
session_3/copper_trade/includes/eu_ets.inc:53:	set_fuel_ttw("methanol_electro", "nitrous_oxide", <Variable("eu_default_diesel_ice_nitrous_oxide") * Forecast("eu_ets_ghg_phase_in")>)
session_3/copper_trade/includes/fuel_eu.inc:34:	set_gwp("carbon_dioxide", 1)
session_3/copper_trade/includes/fuel_eu.inc:35:	set_gwp("methane", 25)
session_3/copper_trade/includes/fuel_eu.inc:36:	set_gwp("nitrous_oxide", 298)
session_3/copper_trade/includes/fuel_eu.inc:60:	set_fuel_ttw("methanol_electro", "carbon_dioxide",

---

### 4. Data about behavioural traits

The most important data about how the owners and the producers decide are:
- the **decision sensitivities**: how sharply the owners respond to a cost difference, and the producers to a difference in expected demand or cost
- the two **inertias**: the owners and the operators (when bunkering) each repeat part of the previous year's choices, as much as the inertia says

<details>
<summary>What else bounds the decisions</summary>

Besides these, the decisions are bounded by limits: how often a ship can be retrofitted (`RetrofitInterval`), how far the operators may change speed (`MaximumSpeedChange`) and how fast the producers can build (`MaximumDevelopment`).

</details>

Each sensitivity says how strongly the owners favour the cheaper option. The
fleet stands for many owners who face the same costs but do not all decide
alike, so a choice comes out as shares spread across them. A value close to 1
means they hardly care about cost and their choices stay evenly spread. A value
far from 1 means nearly all of them pick the cheapest.

<details>
<summary>The formulas: fuel type sensitivity (newbuild and retrofit)</summary>

For the choice of fuel type for a newbuild ship, with $c_i$ the levelised cost per cargo-mile of a new ship of fuel type $i$, $s_i$ its share of the newbuilds and $S$ the value of
`FuelSensitivity`:

$$
\frac{s_i}{s_j} = \left(\frac{c_i}{c_j}\right)^{-\beta},
\qquad \beta = -\frac{\ln S}{\ln 1.1}
$$

Retrofits work the same way, but on NPV rather than cost. When a ship reaches a
retrofit cycle, each possible conversion is weighed against keeping the ship as
it is. With $\mathrm{NPV}_i$ the net present value of conversion $i$ (the fuel and compliance savings over the ship's remaining life, minus the conversion cost), CAPEX the capital cost of a new ship of its type and $S$ the value of `ConversionSensitivity`:

$$
\frac{s_i}{s_\mathrm{keep}} = e^{\beta \, \mathrm{NPV}_i / \mathrm{CAPEX}},
\qquad \beta = \frac{\ln S}{0.05}
$$

</details>

The producers choose their plants the same way, in two steps: the year's build is
first shared between fuels by expected demand (the `Producer`'s `FuelSensitivity`), then each
fuel's share is split between the plants that make it, by delivered cost
(`VariantSensitivity`). [Session 4](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/04-how-navigate-solves.ipynb) shows both steps at work, in
"Stage 3 — The producers decide too".

<details>
<summary>The formulas: fuel demand sensitivity and fuel cost sensitivity</summary>

First, the year's
build is shared between fuels by demand. With $D_i$ the expected unmet demand for
fuel $i$ over a plant's lifetime, discounted so the first years count most, $s_i$
its share of the build and $S$ the value of the `Producer`'s `FuelSensitivity`:

$$
\frac{s_i}{s_j} = \left(\frac{D_i}{D_j}\right)^{\beta},
\qquad \beta = \frac{\ln S}{\ln 1.1}
$$

With the default calibration's 1.25, a fuel with 10 % more expected demand gets 1.25 times the
odds. Then each fuel's share is split between the plants that make it, by cost.
With $c_p$ the levelised cost of the fuel that plant $p$ delivers to the ports, $s_p$
its share and $S$ the value of `VariantSensitivity`:

$$
\frac{s_p}{s_q} = \left(\frac{c_p}{c_q}\right)^{-\beta},
\qquad \beta = -\frac{\ln S}{\ln 1.1}
$$

</details>

In both cases, the sensitivities and the inertias come from `Load DefaultCalibration`. It sets both inertias to 0: each year's choices start afresh. Run the next cell to read it: its comments say what each value means.

In [17]:
# the decision sensitivities and the inertias, which both cases load from the library with Load DefaultCalibration
!cat {LIBRARY}/modules/default_calibration.inc

# SPDX-FileCopyrightText: 2026 Fonden Mærsk Mc-Kinney Møller Center for Zero Carbon Shipping
# SPDX-License-Identifier: CC-BY-4.0

Port "*" {
	set_bunkering_inertia("*", 0.0)
}

# Fleet decision-process calibration.
#
# Each sensitivity is an odds ratio against a fixed reference change. For the cost-based choices
# (lower-is-better levelized cost of transport) a value below 1 penalizes a 10% higher cost; for the
# NPV-based choices (higher-is-better) a value above 1 rewards an NPV advantage equal to 5% of the
# summed ship Capex. A value of 1 means no preference.
Fleet "*" {
	Inertia = 0.0
	FuelSensitivity = 0.75         # a 10% higher cost gives 25% lower odds of a fuel type
	VariantSensitivity = 0.75         # a 10% higher cost gives 25% lower odds of a technology variant
	ConversionSensitivity = 2       # an NPV advantage of 5% of ship Capex doubles the odds
	TechnologySensitivity = 2           # an NPV advantage of 5% of ship Capex doubles the odds
}

# Producer decision-process ca

## Workshop sessions

**Next: [Session 4: How Navigate solves](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/04-how-navigate-solves.ipynb)**

Each link opens the notebook in Google Colab, in a fresh session, so run it
from the top.

1. [Run a simple case](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/dev/docs/notebooks/notebook_1/notebook_1.ipynb)
2. [Run a global case](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/dev/docs/notebooks/notebook_2/notebook_2.ipynb)
3. **How a Navigate model is built** (this session)
4. [How Navigate solves](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/04-how-navigate-solves.ipynb)
5. [Build your own case](https://colab.research.google.com/github/zerocarbonshipping/navigate-zcs/blob/workshop-colab/docs/workshop/05-build-your-own-case.ipynb)